# Colab Setup
Run this notebook top to bottom every session. It restores your work from Drive if a backup exists (so a runtime disconnect doesn't cost you the ~26 min preprocessing run again), otherwise downloads and processes fresh.

In [ ]:
# 1. Mount Google Drive (persists checkpoints/outputs across sessions)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Clone the repo
%cd /content
!git clone https://github.com/jrsheffie/Review-Summarization-LLM.git
%cd Review-Summarization-LLM

In [ ]:
# 3. Install dependencies
!pip install -q -r requirements.txt

In [ ]:
# 4. Confirm GPU runtime is enabled (Runtime -> Change runtime type -> T4 GPU)
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only -- go to Runtime > Change runtime type > GPU')

In [ ]:
# 5. Try restoring raw data + processed data from Drive backup first.
# If no backup exists yet, this just prints a message and does nothing --
# proceed to cells 6-7 to download and process from scratch.
import os

BACKUP_DIR = '/content/drive/MyDrive/Review-Summarization-LLM-backup'
restored = False

if os.path.exists(f'{BACKUP_DIR}/clean_reviews.csv'):
    print('Found backup -- restoring processed data...')
    !mkdir -p data/processed data/raw
    !cp {BACKUP_DIR}/clean_reviews.csv {BACKUP_DIR}/train.csv {BACKUP_DIR}/val.csv {BACKUP_DIR}/test.csv {BACKUP_DIR}/product_batches.json data/processed/ 2>/dev/null
    !cp {BACKUP_DIR}/Reviews.csv data/raw/ 2>/dev/null
    restored = True
    print('Restored from Drive backup.')
else:
    print('No Drive backup found yet -- run the download + preprocessing cells below.')

## Only run cells 6-8 if cell 5 printed "No Drive backup found" -- otherwise skip to cell 9

In [ ]:
# 6. Set up Kaggle API and download dataset (skip if restored above)
# Upload your kaggle.json via the file browser on the left first, then run:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!python data/download_dataset.py

In [ ]:
# 7. Run preprocessing (skip if restored above -- this takes ~20-30 min on the full dataset)
!python data/preprocess.py --input data/raw/Reviews.csv --output-dir data/processed

In [ ]:
# 8. IMPORTANT: back this up to Drive immediately so you never redo steps 6-7 again
!mkdir -p {BACKUP_DIR}
!cp data/processed/*.csv data/processed/*.json {BACKUP_DIR}/
!cp data/raw/Reviews.csv {BACKUP_DIR}/
print('Backed up to Drive.')

## Everyone runs from here down

In [ ]:
# 9. Exploratory data analysis on the real cleaned data
!python data/exploratory_analysis.py --input data/processed/clean_reviews.csv --output-dir outputs/figures

In [ ]:
# 10. BART baseline test -- pretrained model, no fine-tuning yet.
# Proves the model implementation works (Milestone 3 requirement).
from transformers import BartForConditionalGeneration, BartTokenizer
import pandas as pd

tokenizer = BartTokenizer.from_pretrained('facebook/bart-large-cnn')
model = BartForConditionalGeneration.from_pretrained('facebook/bart-large-cnn')

df = pd.read_csv('data/processed/clean_reviews.csv')
sample_reviews = df['Text'].sample(3, random_state=1).tolist()

for i, review in enumerate(sample_reviews):
    inputs = tokenizer(review, return_tensors='pt', max_length=1024, truncation=True)
    summary_ids = model.generate(inputs['input_ids'], num_beams=4, max_length=60, early_stopping=True)
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    print(f'--- Review {i+1} ---')
    print('Original:', review[:200], '...')
    print('BART summary:', summary)
    print()

In [ ]:
# 11. Prompted LLM test -- uses Colab Secrets for the API key (key icon in left sidebar).
# Add a secret named ANTHROPIC_API_KEY, enable notebook access, then run this.
import os, sys, json
from google.colab import userdata

os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
sys.path.insert(0, '.')
from models.llm_prompting import summarize_products

batches = json.load(open('data/processed/product_batches.json'))
results = summarize_products(batches[:3])  # just 3 products as a test

for r in results:
    print(r['product_id'])
    print(r.get('summary') or r.get('error'))
    print('---')